# COVID Feature Engineering

Builds the COVID feature table from covid_ema.csv that will be used in the retraining pipeline.

Pipeline:
  load_covid_ema (filter to clean EMA population)
    -> filter to completed responses
    -> aggregate 9 items to weekly means
    -> build full (student, week) index from label + sensing tables
    -> join COVID items (NaN for pre-COVID weeks and non-respondents)
    -> add covid_period binary flag
    -> save covid_weekly.csv

COVID items included: COVID-1 to COVID-8, COVID-10;
COVID-9 excluded: near-zero correlation with composite score; 
covid_period flag: 0 before 2020-W12, 1 from 2020-W12 onward.

Outputs:
  - data/processed/features/covid_weekly.csv

In [7]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path("../../")))

from src.data.loaders import load_covid_ema, EXCLUDE_UIDS

DATA_DIR    = Path("../../data/raw/college_experience_dataset")
LABEL_PATH  = Path("../../data/processed/labels/weekly_labels.csv")
SENSING_PATH = Path("../../data/processed/features/sensing_weekly.csv")
OUTPUT_DIR  = Path("../../data/processed/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COVID_ITEMS = [f"COVID-{i}" for i in range(1, 11) if i != 9]  
COVID_BOUNDARY = "2020-W12"  # first week of COVID 

print(f"COVID items included: {COVID_ITEMS}")

COVID items included: ['COVID-1', 'COVID-2', 'COVID-3', 'COVID-4', 'COVID-5', 'COVID-6', 'COVID-7', 'COVID-8', 'COVID-10']


---
## Load Reference Population

The full set of (student, week) pairs comes from the union of the label table and sensing table. COVID features must exist as a row for every pair so the assembly join works cleanly.
Pre-COVID weeks and non-respondent students get NaN for COVID items.

In [8]:
labels  = pd.read_csv(LABEL_PATH)
sensing = pd.read_csv(SENSING_PATH)

label_pairs   = labels[["uid", "feature_week"]].rename(columns={"feature_week": "year_week"})
sensing_pairs = sensing[["uid", "year_week"]]

all_pairs = (
    pd.concat([label_pairs, sensing_pairs])
    .drop_duplicates()
    .reset_index(drop=True)
)

clean_uids = all_pairs["uid"].unique().tolist()

print(f"Students in reference population : {len(clean_uids)}")
print(f"Total (student, week) pairs      : {len(all_pairs):,}")
print(f"Unique weeks in reference        : {all_pairs['year_week'].nunique()}")
print(f"Week range: {all_pairs['year_week'].min()} to {all_pairs['year_week'].max()}")

Students in reference population : 216
Total (student, week) pairs      : 28,925
Unique weeks in reference        : 251
Week range: 2017-W36 to 2022-W25


---
## Load COVID EMA

Loads completed COVID EMA responses for the clean population.

In [9]:
COVID_PATH = DATA_DIR / "EMA" / "covid_ema.csv"
df_covid = load_covid_ema(str(COVID_PATH))

completed = df_covid[
    df_covid["has_response"] &
    df_covid["uid"].isin(clean_uids)
].copy()

print(f"Raw COVID EMA rows          : {len(df_covid):,}")
print(f"Completed responses         : {len(completed):,}")
print(f"Students with COVID data    : {completed['uid'].nunique()}")
print(f"Date range                  : {completed['date'].min().date()} "
      f"to {completed['date'].max().date()}")
print(f"\nStudents with NO COVID data : "
      f"{len(clean_uids) - completed['uid'].nunique()}")

Raw COVID EMA rows          : 16,511
Completed responses         : 5,614
Students with COVID data    : 180
Date range                  : 2020-03-18 to 2022-04-23

Students with NO COVID data : 36


---
## Weekly Aggregation

Aggregates COVID items to weekly means per (student, week).
Most students have exactly one COVID survey per week. Mean is used for
consistency with the label and sensing aggregation.

In [10]:
existing_covid = [c for c in COVID_ITEMS if c in completed.columns]
missing_covid  = [c for c in COVID_ITEMS if c not in completed.columns]
if missing_covid:
    print(f"WARNING: COVID items not found in file: {missing_covid}")

covid_weekly = (
    completed.groupby(["uid", "year_week"])[existing_covid]
    .mean()
    .round(4)
    .reset_index()
)

print(f"COVID weekly rows     : {len(covid_weekly):,}")
print(f"Students with data   : {covid_weekly['uid'].nunique()}")
print(f"Weeks with data      : {covid_weekly['year_week'].nunique()}")
print(f"\nSample:")
print(covid_weekly.head(3).to_string(index=False))

COVID weekly rows     : 5,288
Students with data   : 180
Weeks with data      : 110

Sample:
                             uid year_week  COVID-1  COVID-2  COVID-3  COVID-4  COVID-5  COVID-6  COVID-7  COVID-8  COVID-10
003df5deff30e1e5a07b5d063fe85c3f  2020-W16      6.0      6.0      1.0      6.0      6.0      7.0      7.0      7.0       1.0
003df5deff30e1e5a07b5d063fe85c3f  2020-W29      7.0      4.0      1.0      6.0      6.0      7.0      1.0      7.0       1.0
003df5deff30e1e5a07b5d063fe85c3f  2020-W30      4.0      2.0      2.0      5.0      3.0      5.0      2.0      7.0       1.0


---
## Build Full Feature Table

Left joins COVID items onto the full (student, week) reference index.
Pairs without a matching COVID survey get NaN for all COVID items.
This covers:
- All pre-COVID weeks (before 2020-W12)
- Post-COVID weeks where the student did not complete a survey that week
- All weeks for the 36 students who never participated in COVID EMA

In [11]:
covid_full = all_pairs.merge(
    covid_weekly,
    on=["uid", "year_week"],
    how="left"
)

# Add covid_period flag
covid_full["covid_period"] = (covid_full["year_week"] >= COVID_BOUNDARY).astype(int)

print(f"Full COVID feature table:")
print(f"  Rows     : {len(covid_full):,}")
print(f"  Students : {covid_full['uid'].nunique()}")
print(f"  Columns  : {list(covid_full.columns)}")

# NaN breakdown
post_covid = covid_full[covid_full["covid_period"] == 1]
pre_covid  = covid_full[covid_full["covid_period"] == 0]

print(f"\nPre-COVID rows  : {len(pre_covid):,} "
      f"(covid_period=0, all COVID items = NaN by design)")
print(f"Post-COVID rows : {len(post_covid):,} "
      f"(covid_period=1)")

pct_nan_post = post_covid[existing_covid[0]].isna().mean() * 100
print(f"\nPost-COVID rows with NaN COVID items: {pct_nan_post:.1f}%")
print(f"(weeks where student did not complete a COVID survey that week)")

Full COVID feature table:
  Rows     : 28,925
  Students : 216
  Columns  : ['uid', 'year_week', 'COVID-1', 'COVID-2', 'COVID-3', 'COVID-4', 'COVID-5', 'COVID-6', 'COVID-7', 'COVID-8', 'COVID-10', 'covid_period']

Pre-COVID rows  : 16,806 (covid_period=0, all COVID items = NaN by design)
Post-COVID rows : 12,119 (covid_period=1)

Post-COVID rows with NaN COVID items: 57.9%
(weeks where student did not complete a COVID survey that week)


---
## Verification

In [ ]:
# Pre-COVID rows must have covid_period=0 and NaN COVID items
pre = covid_full[covid_full["year_week"] < COVID_BOUNDARY]
n_wrong_period = (pre["covid_period"] != 0).sum()
n_wrong_covid  = pre[existing_covid[0]].notna().sum()

if n_wrong_period > 0:
    print(f"WARNING: {n_wrong_period} pre-COVID rows have covid_period != 0")
else:
    print("Verified: all pre-COVID rows have covid_period = 0")

if n_wrong_covid > 0:
    print(f"WARNING: {n_wrong_covid} pre-COVID rows have non-null COVID items")
else:
    print("Verified: all pre-COVID rows have NaN COVID items")

# Post-COVID rows must have covid_period=1
post = covid_full[covid_full["year_week"] >= COVID_BOUNDARY]
n_wrong = (post["covid_period"] != 1).sum()
if n_wrong > 0:
    print(f"WARNING: {n_wrong} post-COVID rows have covid_period != 1")
else:
    print("Verified: all post-COVID rows have covid_period = 1")

# Students with no COVID data at all
students_with_data = set(covid_weekly["uid"].unique())
students_no_data   = set(clean_uids) - students_with_data
print(f"\nStudents with COVID survey data   : {len(students_with_data)}")
print(f"Students with no COVID survey data: {len(students_no_data)}")
print(f"(These students have NaN for all COVID items in all rows)")

Verified: all pre-COVID rows have covid_period = 0
Verified: all pre-COVID rows have NaN COVID items
Verified: all post-COVID rows have covid_period = 1

Students with COVID survey data   : 180
Students with no COVID survey data: 36
(These students have NaN for all COVID items in all rows)


---
## Save

In [13]:
out_path = OUTPUT_DIR / "covid_weekly.csv"
covid_full.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows  : {len(covid_full):,}")
print(f"Size  : {out_path.stat().st_size / 1024:.1f} KB")
print(f"\nColumns:")
for col in covid_full.columns:
    n_null = covid_full[col].isna().sum()
    pct    = n_null / len(covid_full) * 100
    print(f"  {col}: {pct:.1f}% NaN")

Saved: ..\..\data\processed\features\covid_weekly.csv
Rows  : 28,925
Size  : 1660.5 KB

Columns:
  uid: 0.0% NaN
  year_week: 0.0% NaN
  COVID-1: 82.4% NaN
  COVID-2: 82.4% NaN
  COVID-3: 82.4% NaN
  COVID-4: 82.4% NaN
  COVID-5: 82.4% NaN
  COVID-6: 82.4% NaN
  COVID-7: 82.4% NaN
  COVID-8: 82.4% NaN
  COVID-10: 82.4% NaN
  covid_period: 0.0% NaN
